# LagerNVS Inference

Novel view synthesis from unposed input images. The model does not require input camera poses — only a target camera trajectory specifying where to render from.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f"Device: {device}, dtype: {dtype}")

## 1. Load input images

Provide paths to 1 or more input images. Two preprocessing modes are available:
- `"resize"` (default): Longer side set to `target_size`, aspect ratio preserved, shorter side rounded to a multiple of `patch_size`.
- `"square_crop"`: Center-crop to the largest inscribed square, then resize to `target_size` × `target_size`. Use for 256px models.

In [ ]:
from vggt.utils.load_fn import load_and_preprocess_images

# Change paths below
image_names = [
    "path/to/imageA.png", "path/to/imageB.png", "path/to/imageC.png"
]

# Resize so that longer side = 512, maintaining aspect ratio
images = load_and_preprocess_images(
    image_names, mode="resize", target_size=512, patch_size=8
).to(device).unsqueeze(0)


num_cond_views = len(image_names)
video_length = 100
image_size_hw = (images.shape[-2], images.shape[-1])

print(f"Loaded {num_cond_views} images, shape: {images.shape}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, num_cond_views, figsize=(5 * num_cond_views, 5))
if num_cond_views == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.imshow(images[0][i].permute(1, 2, 0).cpu())
    ax.set_title(f"View {i}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Create target camera path

The model does not require input camera poses. This step automatically constructs a smooth target trajectory by inferring approximate input view positions (using VGGT), then interpolating a path through them.

- **Multi-view (≥2 images):** B-spline interpolation through inferred positions
- **Single-view (1 image):** Forward dolly (+0.3 along camera z-axis)

In [ ]:
from vis import create_target_camera_path

mode = "resize"  # Use "square_crop" for 256px models

rays, cam_tokens = create_target_camera_path(
    image_names, video_length, num_cond_views, image_size_hw, device, dtype, mode=mode
)

print(f"Rays shape: {rays.shape}")
print(f"Camera tokens shape: {cam_tokens.shape}")

## 4. Load model and render

In [ ]:
from huggingface_hub import hf_hub_download
from models.encoder_decoder import EncDec_VitB8
from vis import render_chunked

# Use attention_to_features_type="full_attention" for Re10k model
# Use attention_to_features_type="bidirectional_cross_attention" for General/DL3DV models
model = EncDec_VitB8(pretrained_vggt=False, attention_to_features_type="bidirectional_cross_attention")
ckpt_path = hf_hub_download("facebook/lagernvs_general_512", filename="model.pt")
model.load_state_dict(torch.load(ckpt_path)["model"])
model.to(device)
model.eval()

In [ ]:
print(images.shape, rays.shape, cam_tokens.shape)

In [ ]:
with torch.no_grad():
    with torch.amp.autocast(device_type="cuda", dtype=dtype):
        video_out = render_chunked(
            model,
            (images, rays, cam_tokens),
            num_cond_views=num_cond_views,
            device=device)

print(f"Output video shape: {video_out.shape}")

## 5. Visualize results

In [ ]:
# Show a few sampled frames from the output video
video = video_out[0]  # (V, C, H, W)
num_frames = video.shape[0]
sample_indices = torch.linspace(0, num_frames - 1, 8).long()

fig, axes = plt.subplots(1, len(sample_indices), figsize=(3 * len(sample_indices), 3))
for i, idx in enumerate(sample_indices):
    axes[i].imshow(video[idx].permute(1, 2, 0).clamp(0, 1).cpu())
    axes[i].set_title(f"Frame {idx.item()}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from eval.export import save_video

In [ ]:
from IPython.display import Video, display

output_path = "output_video.mp4"
save_video(video_out[0], output_path)
display(Video(output_path, embed=True))